In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from pyspark.sql.functions import input_file_name, current_timestamp, current_date,col

In [0]:
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "artemzharkov10_bronze")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

In [0]:
LANDING_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/streaming_bitcoin"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/checkpoints/bronze_bitcoin_stream"
SCHEMA_PATH     = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/checkpoints/bronze_bitcoin_schema"
TABLE_NAME      = f"{CATALOG}.{SCHEMA}.stream_bitcoin_bronze"

In [0]:
raw_stream = (
    spark.readStream
    .format("cloudFiles") 
    .option("cloudFiles.schemaHints", 
            "Date STRING, Open STRING,High STRING,Low STRING,Close STRING, Volume STRING") #schema enforcment
    .option("cloudFiles.format", "json") 
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # evolution schema 
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") # creating additional column with error data (except break down all pipline)
    .load(LANDING_PATH)
)

In [0]:
# metadata
df_bronze = (
    raw_stream 
    .withColumn("source_filename", col("_metadata.file_path"))
    .withColumn("ingest_timestamp", current_timestamp())
)

In [0]:
(df_bronze.writeStream
 .format("delta")
 .option("checkpointLocation", CHECKPOINT_PATH)
 .option("mergeSchema", "true") # confirm to save schema with new columns (schema evolution)
 .trigger(availableNow=True) # stop when done
 .table(TABLE_NAME))

In [0]:
df_bronze = spark.table(f"{CATALOG}.{SCHEMA}.stream_bitcoin_bronze")
display(df_bronze)